In [ ]:
"""
=============================================================================
 INFÉRENCE — PIPELINE v5
=============================================================================
 PFE — Trustworthy AI for Opportunistic Screening of Cardiovascular Risk
 Abdallah OUAHMANE — FSR Rabat — Encadrant : Pr. Mohamed EL HASSOUNI

 OBJET
   Exécute le pipeline complet sur 501 images NIH ChestX-ray14 et produit
   le fichier de scores servant de base aux études du Chapitre 6.

 REMARQUE SUR LE NOM
   Ce script réalise une évaluation TECHNIQUE sur données rétrospectives.
   Il ne constitue pas une validation clinique : les étiquettes utilisées
   comme vérité terrain sont des labels radiologiques extraits par NLP, et
   non des événements cardiovasculaires confirmés (voir §6.8.2).



 SEUILS (déterminés empiriquement, §4.6.2)
   FAIBLE  : score < 25 %
   MODÉRÉ  : 25 % ≤ score < 35%
   ÉLEVÉ   : score ≥ 35 %   (maximise l'indice de Youden)

 SORTIE
   resultats_phase4_v5.csv  — scores bruts et fusionnés des 501 images
   pipeline_v5_metriques.png
=============================================================================
"""

import os
import cv2
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve

plt.rcParams['font.family'] = 'DejaVu Sans'

# ============================================================
# CONFIGURATION
# ============================================================
MODELS_DIR = '/kaggle/input/datasets/abdallahouahmane/pfe-cardio-models'
CSV_PATH   = '/kaggle/input/datasets/rahulogoel/clahe-enhancement-on-chestx-ray14/Data_Entry.csv'
IMG_DIR    = '/kaggle/input/datasets/rahulogoel/clahe-enhancement-on-chestx-ray14/IMAGE/IMAGE'

UNET_CARDIO  = f'{MODELS_DIR}/unet_cardio_robust_v2_best.pth'
UNET_CALCIF  = f'{MODELS_DIR}/unet_calcif_best_kagg (2).pth'
CALCIF_MODEL = f'{MODELS_DIR}/calcif_model_final.pth'
RISK_MODEL   = f'{MODELS_DIR}/cardio_risk_v2_best (1).pth'

OUT_CSV = 'resultats_phase4_v5.csv'
OUT_FIG = 'pipeline_v5_metriques.png'

CATEGORIES_RISK   = ['Normal', 'Cardiomegaly', 'Edema']
CATEGORIES_CALCIF = ['Normal', 'Calcification', 'Aortic_enlargement']
N_PAR_CLASSE = 167

# --- Pondération de fusion (Pipeline V5) ---
W_CTR    = 0.15
W_CONG   = 0.40
W_CALCIF = 0.10
W_CARDIO = 0.35

# --- Seuils de catégorisation (§4.6.2) ---
SEUIL_ELEVE  = 35
SEUIL_MODERE = 25

# --- Garde-fou Module 3 : conservé pour traçabilité ---
# M3 est exclu de la fusion, ce seuil n'influence donc plus le score.
# Il reste en place afin de documenter le comportement du module.
SEUIL_CONFIANCE_M3 = 0.60

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("=" * 62)
print("  INFÉRENCE — PIPELINE V3")
print("=" * 62)
print(f"  Device        : {device}")
print(f"  Fusion        : {W_CTR}×s_ctr + {W_CONG}×s_cong")
print(f"  Seuil ÉLEVÉ   : ≥ {SEUIL_ELEVE} %")
print(f"  Seuil MODÉRÉ  : ≥ {SEUIL_MODERE} %")

# ============================================================
# TRANSFORMS
# ============================================================
T512 = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])
T224 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])


# ============================================================
# ARCHITECTURES
# ============================================================
class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super(UNet, self).__init__()

        def double_conv(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c), nn.ReLU(inplace=True)
            )

        self.enc1 = double_conv(in_channels, 64)
        self.enc2 = double_conv(64, 128)
        self.enc3 = double_conv(128, 256)
        self.enc4 = double_conv(256, 512)
        self.pool = nn.MaxPool2d(2, 2)
        self.bottleneck = double_conv(512, 1024)
        self.up4  = nn.ConvTranspose2d(1024, 512, 2, 2)
        self.dec4 = double_conv(1024, 512)
        self.up3  = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec3 = double_conv(512, 256)
        self.up2  = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec2 = double_conv(256, 128)
        self.up1  = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec1 = double_conv(128, 64)
        self.final = nn.Conv2d(64, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([e4, self.up4(b)],  dim=1))
        d3 = self.dec3(torch.cat([e3, self.up3(d4)], dim=1))
        d2 = self.dec2(torch.cat([e2, self.up2(d3)], dim=1))
        d1 = self.dec1(torch.cat([e1, self.up1(d2)], dim=1))
        return torch.sigmoid(self.final(d1))


class EfficientNetCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(EfficientNetCNN, self).__init__()
        self.backbone = models.efficientnet_b0(weights=None)
        orig = self.backbone.features[0][0]
        self.backbone.features[0][0] = nn.Conv2d(
            1, orig.out_channels,
            kernel_size=orig.kernel_size,
            stride=orig.stride,
            padding=orig.padding,
            bias=False
        )
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(in_features, 256),
            nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, num_classes)
        )
        self._activations = None
        self.backbone.features[-1].register_forward_hook(self._save_act)

    def _save_act(self, m, i, o):
        self._activations = o

    def forward(self, x):
        return self.backbone(x)


# ============================================================
# CHARGEMENT DES MODÈLES
# ============================================================
def charger_modeles():
    print("\nChargement des modèles")

    m1 = UNet().to(device)
    m1.load_state_dict(torch.load(UNET_CARDIO, map_location=device))
    m1.eval()
    print("  M1 U-Net cardiaque          : chargement strict OK")

    m2 = UNet().to(device)
    m2.load_state_dict(torch.load(UNET_CALCIF, map_location=device))
    m2.eval()
    print("  M2 U-Net calcifications     : chargement strict OK")

    # ---- Module 3 -------------------------------------------------------
    # Le checkpoint disponible est incompatible avec cette architecture
    # (préfixe de clés, 3 canaux d'entrée, 1000 classes de sortie). Un
    # chargement strict lèverait une exception ; strict=False est conservé
    # pour permettre l'exécution, le module étant exclu de la fusion.
    # Le diagnostic complet figure en Section 6.3 du rapport.
    m3 = EfficientNetCNN(num_classes=3).to(device)
    m3.load_state_dict(torch.load(CALCIF_MODEL, map_location=device))
    m3.eval()
    print("  M3 CNN calcifications       : chargement strict OK")

    m4 = EfficientNetCNN(num_classes=3).to(device)
    m4.load_state_dict(torch.load(RISK_MODEL, map_location=device))
    m4.eval()
    print("  M4 CNN risque               : chargement strict OK")

    return m1, m2, m3, m4


# ============================================================
# CALCUL DU CTR
# ============================================================
def calculer_ctr(mask_cardio, spacing_corrige, position):
    """
    Le CTR est un rapport de deux longueurs converties par le même facteur
    d'échelle : il est donc invariant par changement d'échelle isotrope.
    La conversion en centimètres sert à reporter heart_cm et thorax_cm en
    unités cliniques, non à corriger le ratio (voir §7.4.1).

    Le diamètre thoracique n'est pas segmenté mais approximé à 75 % de la
    largeur de l'image — limite documentée en §7.4.1.
    """
    h, w = mask_cardio.shape
    contours, _ = cv2.findContours(
        mask_cardio.astype(np.uint8),
        cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    heart_px = 0
    if contours:
        c = max(contours, key=cv2.contourArea)
        x, y, w_box, h_box = cv2.boundingRect(c)
        cx, cy = x + w_box // 2, y + h_box // 2
        # Le contour retenu doit être centré dans la zone anatomique
        if w * 0.20 < cx < w * 0.80 and h * 0.20 < cy < h * 0.80:
            heart_px = w_box

    thorax_px = int(w * 0.75)
    heart_cm  = (heart_px  * spacing_corrige) / 10
    thorax_cm = (thorax_px * spacing_corrige) / 10

    ctr_raw = heart_cm / thorax_cm if (thorax_cm > 0 and heart_px > 0) else 0.0
    # Correction de la magnification en incidence antéro-postérieure
    ctr_corrige = ctr_raw - 0.05 if 'AP' in str(position).upper() else ctr_raw

    # Indicateur de plausibilité anatomique (voir §7.4.3)
    ctr_valide = (0.30 <= ctr_corrige <= 0.75)

    return ctr_corrige, heart_cm, thorax_cm, ctr_valide


# ============================================================
# ANALYSE D'UNE IMAGE
# ============================================================
def analyser_image(img_path, m1, m2, m3, m4, df_meta, image_name):
    img_pil = Image.open(img_path).convert('L')

    # --- Métadonnées ---
    spacing    = 0.143
    position   = 'PA'
    label_reel = 'N/A'
    img_orig_w = img_pil.size[0]

    if image_name in df_meta.index:
        row        = df_meta.loc[image_name]
        spacing    = float(row.get('OriginalImagePixelSpacing[x', 0.143))
        position   = str(row.get('View Position', 'PA'))
        label_reel = row.get('Finding Labels', 'N/A')
        img_orig_w = float(row.get('OriginalImage[Width', img_pil.size[0]))

    spacing_corrige = spacing * (img_orig_w / 512.0)

    inp512 = T512(img_pil).unsqueeze(0).to(device)
    inp224 = T224(img_pil).unsqueeze(0).to(device)

    # --- M1 : segmentation cardiaque et CTR ---
    with torch.no_grad():
        mask_cardio = (m1(inp512).cpu().squeeze().numpy() > 0.5).astype(np.uint8)
    ctr, heart_cm, thorax_cm, ctr_valide = calculer_ctr(
        mask_cardio, spacing_corrige, position)
    cardio_flag = ctr > 0.50

    # --- M2 : surface calcifiée (visualisation, hors fusion) ---
    with torch.no_grad():
        mask_calcif = (m2(inp512).cpu().squeeze().numpy() > 0.5).astype(np.uint8)
    calcif_pct = float(mask_calcif.sum()) / mask_calcif.size * 100

    # --- M3 : conservé pour traçabilité, exclu de la fusion ---
    with torch.no_grad():
        prob3 = torch.softmax(m3(inp224), dim=1).cpu().numpy()[0]
    calcif_cls_label = CATEGORIES_CALCIF[int(np.argmax(prob3))]
    confiance_m3 = float(max(prob3))
    m3_utilise = confiance_m3 >= SEUIL_CONFIANCE_M3
    calcif_cls_score = float(prob3[1]) + float(prob3[2])

    # --- M4 : classification du risque ---
    with torch.no_grad():
        prob4 = torch.softmax(m4(inp224), dim=1).cpu().numpy()[0]
    risk_label  = CATEGORIES_RISK[int(np.argmax(prob4))]
    prob_normal = float(prob4[0])
    prob_cardio = float(prob4[1])
    prob_edema  = float(prob4[2])

    # --- Scores normalisés ---
    s_ctr    = min(max((ctr - 0.40) / 0.20, 0.0), 1.0)
    s_calcif = min(calcif_cls_score, 1.0)
    s_cong   = prob_edema

    # --- Fusion Pipeline V3 ---
    risk_global = round((W_CTR * s_ctr + W_CONG * s_cong
                     + W_CALCIF * s_calcif + W_CARDIO * prob_cardio) * 100, 1)

    # --- Catégorisation ---
    if risk_global >= SEUIL_ELEVE:
        niveau = 'ÉLEVÉ'
    elif risk_global >= SEUIL_MODERE:
        niveau = 'MODÉRÉ'
    else:
        niveau = 'FAIBLE'

    # Signalement des segmentations non exploitables (voir §7.4.3)
    if not ctr_valide:
        niveau = 'INDÉTERMINÉ'

    return {
        'image':        image_name,
        'label_reel':   label_reel,
        'ctr':          round(ctr, 3),
        'ctr_valide':   ctr_valide,
        'cardio_flag':  cardio_flag,
        'heart_cm':     round(heart_cm, 2),
        'thorax_cm':    round(thorax_cm, 2),
        'calcif_pct':   round(calcif_pct, 2),
        'calcif_label': calcif_cls_label,
        'confiance_m3': round(confiance_m3, 3),
        'm3_utilise':   m3_utilise,
        'calcif_score': round(calcif_cls_score, 3),
        'prob_normal':  round(prob_normal, 3),
        'prob_cardio':  round(prob_cardio, 3),
        'prob_edema':   round(prob_edema, 3),
        'risk_label':   risk_label,
        'risk_global':  risk_global,
        'niveau':       niveau,
        's_ctr':        round(s_ctr, 3),
        's_calcif':     round(s_calcif, 3),
        's_cong':       round(s_cong, 3),
    }


# ============================================================
# CONSTITUTION DU JEU DE TEST
# ============================================================
def extraire_test_set(csv_path, img_dir, n_par_classe=167):
    """
    Tirage stratifié de n images par classe (seed=99).

    LIMITE : ce tirage ne garantit pas l'exclusion des images ayant servi
    à l'entraînement du Module 4. Un recouvrement est possible et doit être
    quantifié — la liste des identifiants d'entraînement de M4 est requise.
    """
    print("\nConstitution du jeu de test")
    df = pd.read_csv(csv_path)
    df = df[df['Image Index'].apply(
        lambda x: os.path.exists(os.path.join(img_dir, x)))]

    normal = df[df['Finding Labels'] == 'No Finding'] \
        .sample(n=n_par_classe, random_state=99)
    cardio = df[df['Finding Labels'].str.contains('Cardiomegaly', na=False) &
                ~df['Finding Labels'].str.contains('Edema', na=False)] \
        .sample(n=n_par_classe, random_state=99)
    edema = df[df['Finding Labels'].str.contains('Edema', na=False) &
               ~df['Finding Labels'].str.contains('Cardiomegaly', na=False)] \
        .sample(n=n_par_classe, random_state=99)

    test_df = pd.concat([normal, cardio, edema]) \
        .sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"  Normal {len(normal)} | Cardiomegaly {len(cardio)} | "
          f"Edema {len(edema)}  →  total {len(test_df)}")
    return test_df


# ============================================================
# EXÉCUTION DU PIPELINE
# ============================================================
def lancer_pipeline(test_df, m1, m2, m3, m4, df_meta, img_dir):
    print(f"\nAnalyse de {len(test_df)} images")
    resultats, erreurs = [], 0

    for i, (_, row) in enumerate(test_df.iterrows()):
        img_name = row['Image Index']
        img_path = os.path.join(img_dir, img_name)
        if not os.path.exists(img_path):
            continue
        try:
            resultats.append(
                analyser_image(img_path, m1, m2, m3, m4, df_meta, img_name))
        except Exception as e:
            erreurs += 1
            print(f"  Erreur sur {img_name} : {e}")

        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(test_df)}")

    df_res = pd.DataFrame(resultats)
    df_res.to_csv(OUT_CSV, index=False)

    n_m3 = int((~df_res['m3_utilise']).sum())
    n_ctr = int((~df_res['ctr_valide']).sum())
    print(f"\n  {len(df_res)} images analysées, {erreurs} erreurs")
    print(f"  M3 sous le seuil de confiance : {n_m3}/{len(df_res)} "
          f"({n_m3 / len(df_res) * 100:.1f} %)")
    print(f"  CTR hors plage [0.30 ; 0.75]  : {n_ctr}/{len(df_res)} "
          f"({n_ctr / len(df_res) * 100:.1f} %)")
    return df_res


# ============================================================
# MÉTRIQUES
# ============================================================
def calculer_metriques(df_res):
    print("\n" + "=" * 62)
    print("  MÉTRIQUES — PIPELINE V3")
    print("=" * 62)

    y_true = df_res['label_reel'].apply(
        lambda x: 1 if ('Cardiomegaly' in str(x) or 'Edema' in str(x)) else 0
    ).values
    y_prob = (df_res['risk_global'] / 100).values
    y_pred = (df_res['risk_global'] >= SEUIL_ELEVE).astype(int).values

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    vpp  = tp / (tp + fp) if (tp + fp) else 0.0
    vpn  = tn / (tn + fn) if (tn + fn) else 0.0
    acc  = (tp + tn) / len(y_true)
    auc  = roc_auc_score(y_true, y_prob)

    fpr, tpr, seuils = roc_curve(y_true, y_prob)
    seuil_youden = seuils[int(np.argmax(tpr - fpr))] * 100

    print(f"\n  AUC-ROC       : {auc:.4f}")
    print(f"  Au seuil de {SEUIL_ELEVE} % :")
    print(f"    Sensibilité : {sens:.3f}   ({fn} patients manqués)")
    print(f"    Spécificité : {spec:.3f}   ({fp} fausses alertes)")
    print(f"    VPP {vpp:.3f} | VPN {vpn:.3f} | Exactitude {acc:.3f}")
    print(f"    TP={tp}  FP={fp}  TN={tn}  FN={fn}")
    print(f"  Seuil de Youden recalculé : {seuil_youden:.1f} %")

    print("\n  Répartition des niveaux :")
    for niv, n in df_res['niveau'].value_counts().items():
        print(f"    {niv:<13} {n:>4}  ({n / len(df_res) * 100:.1f} %)")

    return dict(auc=auc, sens=sens, spec=spec, vpp=vpp, vpn=vpn, acc=acc,
                tp=tp, fp=fp, tn=tn, fn=fn, fpr=fpr, tpr=tpr,
                seuil_youden=seuil_youden, y_true=y_true, y_prob=y_prob)


# ============================================================
# CONTRÔLE DE REPRODUCTIBILITÉ
# ============================================================
def controler_reproductibilite(df_res, ancien_csv):
    """Les scores bruts doivent être identiques à ceux de l'exécution
    précédente : seule la formule de fusion a changé."""
    if not os.path.exists(ancien_csv):
        print("\n  Contrôle ignoré : ancien CSV introuvable.")
        return

    print("\n" + "=" * 62)
    print("  CONTRÔLE DE REPRODUCTIBILITÉ")
    print("=" * 62)
    old = pd.read_csv(ancien_csv).set_index('image')
    new = df_res.set_index('image')
    communs = old.index.intersection(new.index)
    print(f"  Images communes : {len(communs)}")

    for col in ['ctr', 's_ctr', 's_cong', 'prob_edema', 'prob_normal']:
        if col in old.columns and col in new.columns:
            ecart = (old.loc[communs, col] - new.loc[communs, col]).abs().max()
            statut = "identique" if ecart < 1e-9 else f"ÉCART max {ecart:.2e}"
            print(f"    {col:<13} {statut}")

    if 'risk_global' in old.columns:
        auc_v1 = roc_auc_score(
            old.loc[communs, 'label_reel'].apply(
                lambda x: 1 if ('Cardiomegaly' in str(x) or 'Edema' in str(x)) else 0),
            old.loc[communs, 'risk_global'])
        print(f"\n  AUC de l'ancien risk_global (formule V1) : {auc_v1:.4f}")


# ============================================================
# FIGURE
# ============================================================
def visualiser(df_res, m):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))

    ax = axes[0]
    ax.plot(m['fpr'], m['tpr'], lw=2.5, color='#1E8449',
            label=f"Pipeline V3 — AUC = {m['auc']:.3f}")
    ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Hasard')
    idx = int(np.argmax(m['tpr'] - m['fpr']))
    ax.scatter(m['fpr'][idx], m['tpr'][idx], color='#C0392B', s=90, zorder=5,
               label=f"Youden ({m['seuil_youden']:.0f} %)")
    ax.set_xlabel('1 − Spécificité')
    ax.set_ylabel('Sensibilité')
    ax.set_title('Courbe ROC — Pipeline V3', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

    ax = axes[1]
    cm = np.array([[m['tn'], m['fp']], [m['fn'], m['tp']]])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Prédit négatif', 'Prédit positif'],
                yticklabels=['Réel négatif', 'Réel positif'],
                annot_kws={'size': 17, 'weight': 'bold'})
    ax.set_title(f"Matrice de confusion — seuil {SEUIL_ELEVE} %\n"
                 f"Sensibilité {m['sens']:.3f} | Spécificité {m['spec']:.3f}",
                 fontweight='bold')

    ax = axes[2]
    groupes = {
        'Normal': df_res[df_res['label_reel'] == 'No Finding']['risk_global'],
        'Cardiomegaly': df_res[df_res['label_reel']
                               .str.contains('Cardiomegaly', na=False)]['risk_global'],
        'Edema': df_res[df_res['label_reel']
                        .str.contains('Edema', na=False)]['risk_global'],
    }
    bp = ax.boxplot([v.values for v in groupes.values()],
                    tick_labels=list(groupes.keys()),
                    patch_artist=True, notch=True)
    for patch, c in zip(bp['boxes'], ['#2ECC71', '#E74C3C', '#F39C12']):
        patch.set_facecolor(c)
        patch.set_alpha(0.7)
    ax.axhline(SEUIL_ELEVE, color='#C0392B', ls='--', lw=1.8,
               label=f'ÉLEVÉ ({SEUIL_ELEVE} %)')
    ax.axhline(SEUIL_MODERE, color='#F39C12', ls='--', lw=1.8,
               label=f'MODÉRÉ ({SEUIL_MODERE} %)')
    ax.set_ylabel('risk_global (%)')
    ax.set_title('Distribution des scores par classe', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3, axis='y')

    plt.tight_layout()
    for ext in ('png', 'svg'):
        plt.savefig(OUT_FIG.replace('.png', f'.{ext}'), dpi=300,
                    bbox_inches='tight', facecolor='white')
    print(f"\n  Figure enregistrée : {OUT_FIG}")


# ============================================================
# PROGRAMME PRINCIPAL
# ============================================================
if __name__ == '__main__':
    m1, m2, m3, m4 = charger_modeles()

    df_meta = pd.read_csv(CSV_PATH).set_index('Image Index')
    print(f"\nMétadonnées : {len(df_meta)} entrées")

    test_df = extraire_test_set(CSV_PATH, IMG_DIR, N_PAR_CLASSE)
    df_res = lancer_pipeline(test_df, m1, m2, m3, m4, df_meta, IMG_DIR)

    metriques = calculer_metriques(df_res)

    controler_reproductibilite(
        df_res,
        '/kaggle/input/datasets/abdallahouahmane/resultas-phase4/resultats_phase4_v5.csv')

    visualiser(df_res, metriques)

    print(f"\n  Fichier de sortie : {OUT_CSV}")
    print("  Terminé.")